In [0]:
import dlt as dp
from pyspark.sql import functions as F
from pyspark.sql.functions import col, expr, current_timestamp, when, trim, coalesce, lit

@dp.table(
    name="dbr_dev.skybound_silver.flights_silver",
    table_properties={
        "quality": "silver",
        "pipelines.autoOptimize.zOrderCols": "icao24"
    }
)
def flights_silver():
    return (
        spark.read.table("dbr_dev.skybound_bronze.flights_bronze")
        .withColumn("_ts", col("ingestion_timestamp").cast("timestamp"))
        .where((col("latitude") != "") & (col("longitude") != ""))
        .select(
            col("icao24"),
            col("callsign"),
            col("origin_country"),
            col("longitude").cast("double"),
            col("latitude").cast("double"),
            col("geo_altitude").cast("double").alias("altitude"),
            col("velocity").cast("double").alias("speed"),
            col("true_track").cast("double").alias("heading"),
            col("vertical_rate").cast("double"),
            col("on_ground").cast("boolean"),
            col("_ts").alias("last_seen")
        )
        .withColumn("rn", expr("row_number() over (partition by icao24 order by last_seen desc)"))
        .filter("rn = 1")
        .drop("rn")
    )